# Week 6 — Validation & Research Claim Audit

Lane: Refresh / Content Opportunity Scoring

This notebook does two things: (1) audits two findings from FlyRank's *State of AI-Driven SEO* (March 2026) paper the way we audited it live, and (2) turns that same scrutiny on my own Week-5 model — an honest-split before/after, a leakage audit, and a rewrite of any claims that went further than the evidence.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib
!git clone https://github.com/tanjumnaher01-spec/Starter-Notebooks-FlyrankAI.git
%cd Starter-Notebooks-FlyrankAI
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.inspection import permutation_importance
df = pd.read_csv("./data/raw/content_refresh_anonymized.csv")
df.shape

Cloning into 'Starter-Notebooks-FlyrankAI'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 141 (delta 54), reused 31 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 1.75 MiB | 7.94 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/Starter-Notebooks-FlyrankAI


(30000, 44)

## 1) Two paper findings + my methodology questions

The paper is public-safe and mostly disclosed already — direct aggregate comparisons lead, ML pages are explicitly labeled exploratory, and several sections (Finding #4, Finding #10) flag their own weak spots. My job here isn't to grade it; it's to ask the same questions I'd want asked of my own Week-5 work.

### Finding #10 — AI Model Performance (OpenAI vs Gemini, age-controlled)

**Claim:** Once age mix is controlled, "Gemini leads some cohorts and OpenAI leads others" — no blanket winner.

**My methodology question:** Where does "age-controlled" actually cut the confound? Age tier is a coarse bucket (0-14, 15-30, 31-90 ... 365+), and provider adoption likely shifted over the same 6-month window the trend chart shows (Oct '25 → Mar '26). If OpenAI vs. Gemini usage share also changed month-to-month, age tier bucketing doesn't fully separate "which model" from "when it was published, and what else changed editorially that month." The paper is careful to call this "exploratory," which is the right caveat — but I'd want to see per-tier sample sizes before trusting a cohort comparison this granular.

**Does the validation design support the claim?** Partially. It's a real cohort split, not just a raw average, which is the correct instinct. But it's still observational — no held-out test set, no cross-validation, just a comparison across buckets. That's fine for a "this needs more investigation" claim (which is exactly how the paper frames it), and would not be fine as a "Gemini wins" or "OpenAI wins" headline.

### Finding #4 — The Freshness Multiplier (361+ day bucket, 283:1 ratio)

**Claim:** Growth-to-decline ratio at 31-90 days freshness is 7.88:1 (the paper's real headline number); the 361+ bucket shows 283:1 but is called out as unstable.

**My methodology question:** Where does the label (growing vs. declining) come from for that 361+ bucket specifically? The paper discloses the denominator directly — "283 growing pages versus only 1 declining" — which is exactly the right disclosure. My question is whether that 1 declining page is a real signal or noise: with n=1 in the denominator, a single mislabeled or borderline row flips the ratio by orders of magnitude. This is the textbook small-sample-bucket problem.

**Does the validation design support the claim?** No — and the paper says so itself ("too small and too unstable to treat as a headline multiplier"). This is a good example of a paper being honest about a number it *could* have sold as a headline stat but didn't. The 31-90 day number (7.88:1, much larger sample) is the one that should actually inform action, and that's what the paper's own playbook uses.

## 2) My model under an honest split — before / after

My Week-5 model already used a **client-grouped** split (`GroupShuffleSplit` on `client_id`). The "before" I never actually tested was: what would the same features and model look like under a naive **random row split** — where multiple content items from the same client end up in both train and test? That's the before/after this section runs.

In [2]:
# Rebuild the same target + leakage-safe feature set as Week 5
df = df.dropna(subset=["trend_direction"]).copy()
df["needs_refresh"] = (df["trend_direction"] == "down").astype(int)

leaky_cols = ["trend_direction", "trend_pct", "score", "reason_code", "action",
              "impressions_last_30d","clicks_last_30d","sessions_last_30d",
              "impressions_prev_30d","clicks_prev_30d","sessions_prev_30d"]
id_cols = ["content_id","client_id"]

feature_cols = [c for c in df.columns if c not in leaky_cols + id_cols + ["needs_refresh"]]
X = df[feature_cols].copy()
y = df["needs_refresh"]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
X[cat_cols] = X[cat_cols].astype("category")
for c in cat_cols:
    X[c] = X[c].cat.codes
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

In [3]:
def fit_eval(X_train, X_test, y_train, y_test, label):
    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return {
        "split": label,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
    }

# BEFORE: naive random split — ignores that several rows share the same client_id
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
before = fit_eval(Xr_train, Xr_test, yr_train, yr_test, "Random split (naive, BEFORE)")

# AFTER: client-grouped split — same design as Week 5
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
Xg_train, Xg_test = X.iloc[train_idx], X.iloc[test_idx]
yg_train, yg_test = y.iloc[train_idx], y.iloc[test_idx]
after = fit_eval(Xg_train, Xg_test, yg_train, yg_test, "Client-grouped split (Week-5 design, AFTER)")

comparison = pd.DataFrame([before, after]).set_index("split")
comparison

,accuracy,precision,recall,f1
split,,,,
"Random split (naive, BEFORE)",0.687200,0.689359,0.769742,0.727336
"Client-grouped split (Week-5 design, AFTER)",0.589881,0.583683,0.718367,0.644060


**Interpretation:** If the random split scores noticeably higher than the grouped split, that gap is the client-leakage effect — the model was partly "recognizing" a client's overall pattern rather than generalizing to a new one. The grouped-split numbers (F1 ≈ 0.644 on the original Week-5 run) are the honest, decision-support-safe numbers; the random-split numbers should **not** be reported as the model's real performance, even though they look better on paper.

## 3) Leakage audit

**Features excluded and why (same as Week 5):**
- `trend_direction`, `trend_pct` — these directly define the label (`needs_refresh`); including them is using the answer as an input.
- `impressions_last_30d` / `clicks_last_30d` / `sessions_last_30d` and their `_prev_30d` counterparts — these are the raw components `trend_direction` is computed from. Even without the label column itself, these give the model a near-exact reconstruction of the label.
- `score`, `reason_code`, `action` — these are outputs of the Week-4 hand-written rule, not independent evidence; the rule itself partially encodes `days_since_last_update` and `avg_position`, which are already in the feature set, so keeping these would double-count and blur what the model is actually learning from.
- `content_id`, `client_id` — identifiers, not signal; `client_id` is used only for the group split, never as a feature.

**Demonstration — what happens if a leaky feature is added back in:**

In [4]:
# Deliberately reintroduce one leaky feature to show the effect, mirroring the
# capstone's chg_future leakage test (accuracy jumped to a perfect 1.000 there).
X_leaky = X.copy()
X_leaky["trend_pct_LEAKY"] = df["trend_pct"].fillna(df["trend_pct"].median())

Xl_train, Xl_test = X_leaky.iloc[train_idx], X_leaky.iloc[test_idx]
leaky_result = fit_eval(Xl_train, Xl_test, yg_train, yg_test, "Grouped split + trend_pct leaked back in")

pd.concat([comparison, pd.DataFrame([leaky_result]).set_index("split")])

,accuracy,precision,recall,f1
split,,,,
"Random split (naive, BEFORE)",0.687200,0.689359,0.769742,0.727336
"Client-grouped split (Week-5 design, AFTER)",0.589881,0.583683,0.718367,0.644060
Grouped split + trend_pct leaked back in,1.000000,1.000000,1.000000,1.000000


**Result:** adding `trend_pct` back in should push F1 far above the honest 0.644 — because `trend_pct` is the near-exact numeric source of the `trend_direction` label. This is the same failure mode as the capstone's `chg_future` test (which hit a perfect 1.000 accuracy) — a clean, visible signal that a feature is leaking the answer rather than predicting it.

## 4) Claim rewrite

Rewriting my own Week-5 conclusions in safe, public-facing language — no claim goes further than a held-out, client-grouped comparison actually supports.

| Original (Week-5) | Rewritten (safe) |
|---|---|
| "Gradient Boosting won: F1 0.644 vs baseline's 0.041" | Gradient Boosting **showed a higher measured F1** (0.644) than the Week-4 rule-based baseline (0.041) on a held-out, client-grouped test split — directional evidence the learned model separates decliners better than the hand-written rule, not proof it will generalize to new clients at the same margin. |
| "the rule-based baseline misses ~98% of actually-declining content" | On this held-out sample, the baseline's **measured recall was 0.022** — it flagged very few of the content items later labeled as declining. This is a decision-support observation about this dataset, not a universal claim about rule-based scoring. |
| "Top permutation-importance features: days_with_impressions (dominant)..." | `days_with_impressions` had the **largest measured permutation-importance value** in this model run; this describes what the model leaned on for this dataset and split, not a causal driver of content decline. |
| "False positives... the model over-flags" | In the held-out set, the model's false positives were **concentrated among** content with borderline `avg_position` and stable `trend_direction` — an observed pattern in this error sample, not a general claim about the model's behavior on unseen data. |

## 5) Self-check

- [x] Two paper findings summarized with a concrete, respectful methodology question each (label provenance, validation design).
- [x] My Week-5 model re-run under both a naive random split (before) and the client-grouped split (after, Week-5's actual design) — same features, same model, same target.
- [x] Leakage audit lists every excluded feature with a reason, plus a live demonstration of the effect of leaking one back in.
- [x] Every claim carried over from Week-5 rewritten using observed / measured / directional / decision-support language.
- [x] No dataset files committed — only this notebook, per `DATA_USE.md` and the repo's CI leak-guard.